# Imports Needed

In [1]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 13.4 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 28.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 28.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import pandas as pd
import spacy
import re
import matplotlib.pyplot as plt

# Loading Data

In [3]:
df = pd.read_csv("C:\\Users\\Carol\\AI-Hallucinations-Detection\\data\\cleaned_data.csv")

df.head()

,reference,input,output,label,hallucination_type_realized,question_type,hallucination_type_encouraged
0,"A.D.A.M., Inc. estÃ¡ acreditada por la URAC, t...","What organization has accredited A.D.A.M., Inc...","A.D.A.M., Inc. is accredited by the Atlantis H...",hallucinated,Entity-error hallucination,Default question type,Entity-error hallucination
1,"""This dataset for NOAA's Science On a Sphere d...",What type of educational activity is encourage...,The dataset encourages learners to complete a ...,hallucinated,Relation-error hallucination,Default question type,Relation-error hallucination
2,"Dimension items include ""Foreground"" and ""Back...",What is the primary reason for a hit to be cla...,The primary reason for a hit to be classified ...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination
3,"Atlas Search runs a new process, called mongot...",What are the specific hardware requirements fo...,"Based on our production monitoring data, mongo...",hallucinated,Unverifiable information hallucination,Other common hallucinated questions,Other hallucination
4,The business implications are stark. In a surv...,What percentage of banking executives in the l...,34% of banking executives in the loan originat...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination


# Named Entity and Numerical Fact Overlap Functions

In [4]:
#model we are using (reads people, organizations, locations, numbers, dates, etc.)
nlp = spacy.load("en_core_web_sm")

c:\Users\Carol\miniconda3\envs\spark-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


| Disabled | Reason | Example |
|----------|----------|----------|
| parser | It figures out the sentence structure, but we did not need this in our NER model | Subject, object, relationships between words |
| tagger | Labels words as noun, verb, adjective, etc. which is not needed for entity extraction. Good for grammar tasks though.| "Obama" -> PROPN (proper noun)|
| lemmatizer | Converts words to base form. Feels irrelevant for entity detection. | "running" -> run |

en_core_web_trf was taking quite awhile to run in our for loop when running the model with our data which we theorized that it was taking up too much GB in our RAM

In [5]:
#run twice: input reference column, and input repsonse column
def extract_entities_batch(texts):
    """Extract entities for a list of texts using spaCy's nlp.pipe for efficiency."""
    ner_list = []
    for doc in nlp.pipe(texts, disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"], n_process=-1):
        entities = {}
        for ent in doc.ents:
            entities.setdefault(ent.label_, set()).add(ent.text.lower())
        ner_list.append(entities)
    return ner_list

In [6]:
def flatten_entities(ent_dict):
    """Flatten a dictionary of entity types to a single set of unique entities.
    Args:
        ent_dict (dict): A dictionary where keys are entity types and values are sets of entities.
    Returns:
        set: A set of unique entities."""
    
    s = set()
    for ents in ent_dict.values():
        s.update(ents)
    return s

We previously had {
  "PERSON": {"barack obama"},
  "GPE": {"paris"}
}, now we have {"barack obama", "paris"} after flattening it.

In [7]:
#testing how the function works
texts = ["Barack Obama was president.",
    "Apple is based in Cupertino."]

test = extract_entities_batch(texts)

print(test)

[{'PERSON': {'barack obama'}}, {'ORG': {'apple'}, 'GPE': {'cupertino'}}]


In [8]:
flattened_test = [flatten_entities(ent_dict) for ent_dict in test]
print(flattened_test)

[{'barack obama'}, {'apple', 'cupertino'}]


Later on in our for loop we do "ref_set & resp_set." This would only work cleanly when both sides have something like this {"barack obama", "paris"} instead of {"PERSON": {...}, "GPE": {...}}. Python would try to compare the dictionary structure or keys, not the actual entity strings. We only care about whether the entity strings are in the reference, response, or both.

In [ ]:
def calculate_entity_recall(true_entities, pred_entities):
    """Args: 
    true_entities (set): set of tuples representing the true entities in the reference text.
    pred_entities (set): set of tuples representing the predicted entities in the response text.
    Returns:
        float: The recall score for the given entities."""
    
    #Spans predicted by the model that exists in the reference text (ground truth)
    tp = len(true_entities & pred_entities)

    #Spans in the reference text that were not predicted by the model (missed entities)
    fn = len(true_entities - pred_entities)

    if tp == 0:
        return 1.0

    return tp / len(true_entities)

In [10]:
#out of reference function
def out_of_reference_rate(ref_entities, resp_entities):
    """Args:
    ref_entities (dict): A dictionary where keys are entity types and values are sets of entities extracted from the reference text.
    resp_entities (dict): A dictionary where keys are entity types and values are sets of entities extracted from the response text.
    Returns:
        float: The out-of-reference rate for the given entities."""
    ref_set = set()
    resp_set = set()

    for ents in ref_entities.values():
        ref_set.update(ents)

    for ents in resp_entities.values():
        resp_set.update(ents)

    if len(resp_set) == 0:
        return 0.0

    return len(resp_set - ref_set) / len(resp_set) #the response has that amount of entities that are not in the reference

Looking to see which spacy NER are numerical labels

In [11]:
unknown_explanation = ["CARDINAL", "FAC", "GPE", "NORP", "LAW", "NORP", "ORDINAL","PRODUCT", "WORK_OF_ART"]

for ent in unknown_explanation:
    print(f"{ent}: {spacy.explain(ent)}")

CARDINAL: Numerals that do not fall under another type
FAC: Buildings, airports, highways, bridges, etc.
GPE: Countries, cities, states
NORP: Nationalities or religious or political groups
LAW: Named documents made into laws.
NORP: Nationalities or religious or political groups
ORDINAL: "first", "second", etc.
PRODUCT: Objects, vehicles, foods, etc. (not services)
WORK_OF_ART: Titles of books, songs, etc.


In [12]:
NUMERIC_LABELS = {
    "CARDINAL",
    "ORDINAL",
    "QUANTITY",
    "MONEY",
    "PERCENT",
    "DATE",
    "TIME",}

def number_overlap(ref_entities, resp_entities):
    """Args:
    ref_entities (dict): A dictionary where keys are entity types and values are sets of entities extracted from the reference text.
    resp_entities (dict): A dictionary where keys are entity types and values are sets of entities extracted from the response text.
    Returns:
        float: The overlap rate for the given numerical entities."""
    ref_nums = set()
    resp_nums = set()

    for label in NUMERIC_LABELS:
        ref_nums.update(ref_entities.get(label, set()))
        resp_nums.update(resp_entities.get(label, set()))

    if len(ref_nums) == 0:
        return 1.0

    return len(ref_nums & resp_nums) / len(ref_nums)

# Applying model to our data

In [13]:
print(df.columns)

Index(['reference', 'input', 'output', 'label', 'hallucination_type_realized',
       'question_type', 'hallucination_type_encouraged'],
      dtype='object')


In [21]:
# Extract entities for all texts in batches
ref_entities = extract_entities_batch(df["reference"])
resp_entities = extract_entities_batch(df["output"])

# Flatten entity dictionaries to sets of unique entities
ref_sets = [flatten_entities(x) for x in ref_entities]
resp_sets = [flatten_entities(x) for x in resp_entities]

In [22]:
features = [
    {
        "entity_recall": (
            len(r & s) / len(r) if len(r) > 0 else 1.0
        ),

        "out_of_reference_rate": (
            len(s - r) / len(s) if len(s) > 0 else 0.0
        ),

        "number_overlap": number_overlap(ref_ents, resp_ents)
    }
    for r, s, ref_ents, resp_ents in zip(
        ref_sets,
        resp_sets,
        ref_entities,
        resp_entities
    )
]

In [23]:
feature_df = pd.DataFrame(features)

feature_df.head()

,entity_recall,out_of_reference_rate,number_overlap
0,0.000000,1.000000,1.0
1,0.000000,1.000000,0.0
2,0.500000,0.000000,1.0
3,0.250000,0.900000,1.0
4,0.333333,0.666667,0.5


In [24]:
#To add the label column to the feature dataframe to see what the features look like for hallucinated vs non-hallucinated examples
feature_df["label"] = df["label"]

feature_df.head()

,entity_recall,out_of_reference_rate,number_overlap,label
0,0.000000,1.000000,1.0,hallucinated
1,0.000000,1.000000,0.0,hallucinated
2,0.500000,0.000000,1.0,hallucinated
3,0.250000,0.900000,1.0,hallucinated
4,0.333333,0.666667,0.5,hallucinated


# Observations

| Feature | Low Value (close to 0) | High Value (close to 1) |
|----------|----------|----------|
| entity_recall | Response kept very few reference entities | Response kept most reference entities |
| out_of_reference_rate | Response introduced few new entities | Response introduced many new entities |
| number_overlap | Response changed or omitted most numbers | Response preserved most numbers |

In [25]:
feature_df.groupby("label").mean(numeric_only=True)

,entity_recall,out_of_reference_rate,number_overlap
label,,,
factual,0.351255,0.150352,0.629826
hallucinated,0.388684,0.464771,0.672771


In our dataset, hallucinated responses showcases higher average entity recall, out-of-reference entity rates, and higher numerical overlap than factual responses. This could mean that our hallucinated responses can still repeat many correct entities and numbers while at the same time adding false information.